# Does changing the coach win more points? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/does-firing-the-coach-work/story_001_coach_firing.ipynb)

This R notebook recomputes every statistic in
[Does changing the coach win more points?](https://pointzerofive.net/2026/09/does-firing-the-coach-work),
story #001 on *pointzerofive*, from the raw match files. Colab opens it with the R runtime; run it
top to bottom (`Runtime → Run all`). It runs in a few seconds.

**The question:** when a Bundesliga club changes coach mid-season, does the team get better,
or would it have recovered anyway?

Each quoted number is printed and checked with `stopifnot()` against the published dataset
`data/does-firing-the-coach-work.json`. Requires base R and `jsonlite` (preinstalled in Colab).
The pipeline that wrote the dataset is `notebooks/coach_firing_pipeline.R`; its console output,
with R and package versions, is `data/does-firing-the-coach-work.output.txt`.

In [ ]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/does-firing-the-coach-work/"
REPO <- ".."

# A file of this story: next to the notebook (pointzerofive-data), in the website's repository,
# else downloaded from pointzerofive-data.
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}

near <- function(a, b, tol = 0.0015) stopifnot(abs(a - b) <= tol)

read_csv_utf8 <- function(path) {
  read.csv(path, encoding = "UTF-8", stringsAsFactors = FALSE, check.names = FALSE)
}

RAW <- fetch_path("notebooks/raw/bundesliga/coach_games_L1.csv")
games <- read_csv_utf8(RAW)
matches <- read_csv_utf8(fetch_path("notebooks/raw/bundesliga/matches.csv"))
pub <- fromJSON(fetch_path("data/does-firing-the-coach-work.json"), simplifyVector = FALSE)
cat(sprintf("%d Bundesliga matches with managers, seasons %d–%d\n",
            nrow(games), min(games$season), max(games$season)))
head(games)

## 1. Two sources, one set of results

`coach_games_L1.csv` comes from the transfermarkt-datasets `games` table (CC0 1.0, data from
Transfermarkt.de). `matches.csv` is parsed from openfootball (CC0 1.0) by
`notebooks/bundesliga_matches_pipeline.R`. Team names differ between the two, so we compare, for
each match date, the list of scorelines.

In [ ]:
by_date <- function(df, h, a) {
  lines <- paste(df[[h]], df[[a]], sep = "-")
  tapply(lines, df$date, function(x) paste(sort(x, method = "radix"), collapse = " "))
}
tm <- by_date(games, "home_club_goals", "away_club_goals")
of <- by_date(matches[matches$date %in% names(tm), ], "home_goals", "away_goals")
stopifnot(setequal(names(tm), names(of)), all(tm == of[names(tm)]))
cat(sprintf("all %d match dates agree\n", length(tm)))

## 2. Finding coach changes

One row per team per game, in date order within each season. A **coach change** is the first
game under a different manager. If the previous manager comes back later in the same season,
it was a stand-in (suspension, illness) and does not count. A caretaker followed by a new
permanent coach counts as one change. The column `adj` (points above an expectation from the
opponent's strength) is used only by the "opponents = yes" paths.

In [ ]:
K <- 6L

# Points above expectation given the opponent (spec §7.1, opponents=yes).
#
# Expected points come from a league-wide linear fit of points on the opponent's season points
# per game (this match excluded) and home advantage. The opponent's season figure includes games
# played after a coach change: leakage of later information, accepted and disclosed.
add_expected_points <- function(tg) {
  key <- paste(tg$season, tg$club)
  sums <- tapply(tg$pts, key, sum)
  sizes <- tapply(tg$pts, key, length)
  okey <- paste(tg$season, tg$opp)
  opp_pts_here <- c(3, 1, 0)[match(tg$pts, c(0, 1, 3))]
  tg$opp_ppg_loo <- (sums[okey] - opp_pts_here) / (sizes[okey] - 1)
  x <- cbind(1, tg$opp_ppg_loo, tg$home)
  beta <- qr.coef(qr(x), tg$pts)
  tg$adj <- tg$pts - drop(x %*% beta)
  tg
}

# One row per team per game, in date order within each season.
team_games <- function() {
  l1 <- read_csv_utf8(RAW)
  home <- data.frame(season = l1$season, date = l1$date, home = 1L, club = l1$home_club_name,
                     opp = l1$away_club_name, gf = l1$home_club_goals,
                     ga = l1$away_club_goals, mgr = l1$home_club_manager_name)
  away <- data.frame(season = l1$season, date = l1$date, home = 0L, club = l1$away_club_name,
                     opp = l1$home_club_name, gf = l1$away_club_goals,
                     ga = l1$home_club_goals, mgr = l1$away_club_manager_name)
  tg <- rbind(home, away)
  tg <- tg[order(tg$season, tg$club, tg$date, method = "radix"), ]
  rownames(tg) <- NULL
  tg$pts <- ifelse(tg$gf > tg$ga, 3L, ifelse(tg$gf == tg$ga, 1L, 0L))
  key <- paste(tg$season, tg$club)
  tg$n <- ave(seq_along(key), key, FUN = seq_along)
  season_ppg <- tapply(tg$pts, key, mean)
  tg$opp_ppg <- unname(season_ppg[paste(tg$season, tg$opp)])
  add_expected_points(tg)
}

find_changes <- function(tg, k) {
  changes <- list()
  standins <- 0L
  key <- paste(tg$season, tg$club, sep = "\r")
  for (grp in unique(key)) {
    d <- tg[key == grp, ]
    m <- d$mgr
    n <- length(m)
    i <- 1L  # 0-based position of the game being looked at, as in the original Python
    while (i < n) {
      if (m[i + 1] == m[i]) {
        i <- i + 1L
        next
      }
      outgoing <- m[i]
      rest <- m[(i + 1):n]
      if (outgoing %in% rest) {
        # Stand-in: skip to the outgoing manager's return
        standins <- standins + 1L
        i <- i + match(outgoing, rest)
        next
      }
      permanent <- m[min(i + k - 1L, n - 1L) + 1L]
      changes[[length(changes) + 1]] <- data.frame(
        season = d$season[1], club = d$club[1], game = i + 1L,
        out = outgoing, `in` = m[i + 1], permanent = permanent, check.names = FALSE
      )
      i <- i + k  # a caretaker-then-permanent sequence is one change
    }
  }
  list(changes = do.call(rbind, changes), standins = standins)
}

tg <- team_games()
cat(sprintf("league average: %.3f points per game\n", mean(tg$pts)))
near(round(mean(tg$pts), 3), pub$summary$leaguePpg)
fc <- find_changes(tg, K)
changes <- fc$changes
cat(sprintf("%d coach changes, %d stand-ins excluded\n", nrow(changes), fc$standins))
stopifnot(nrow(changes) == pub$summary$changes, fc$standins == pub$summary$standins)
head(changes)

## 3. Six games before, six games after

For every team-game we sum points over the previous six games (`pre`) and over the next six
including the current one (`post`). Changes in the first or last six games of a season have
an incomplete window and drop out. Control stretches have no coach change in the twelve games
around them.

In [ ]:
# Sum of the k values before position i (NA without k of them).
roll_prev <- function(x, k) {
  n <- length(x)
  out <- rep(NA_real_, n)
  if (n > k) for (i in (k + 1):n) out[i] <- sum(x[(i - k):(i - 1)])
  out
}

# Sum of the value at i and the k - 1 after it (NA without k of them).
roll_next <- function(x, k) {
  n <- length(x)
  out <- rep(NA_real_, n)
  if (n >= k) for (i in 1:(n - k + 1)) out[i] <- sum(x[i:(i + k - 1)])
  out
}

windows <- function(tg, k) {
  key <- paste(tg$season, tg$club)
  by_team <- function(x, f) ave(as.numeric(x), key, FUN = function(v) f(v, k))
  tg$pre <- by_team(tg$pts, roll_prev)
  tg$post <- by_team(tg$pts, roll_next)
  tg$pre_adj <- by_team(tg$adj, roll_prev)
  tg$post_adj <- by_team(tg$adj, roll_next)
  tg$opp_post <- by_team(tg$opp_ppg, roll_next) / k
  tg
}

split_windows <- function(tg, changes, k) {
  w <- windows(tg, k)
  wkey <- paste(w$season, w$club, w$n)
  idx <- match(paste(changes$season, changes$club, changes$game), wkey)
  treated <- cbind(changes, w[idx, setdiff(names(w), c("season", "club"))])
  treated <- treated[!is.na(treated$pre) & !is.na(treated$post), ]
  rownames(treated) <- NULL
  pool <- w[!is.na(w$pre) & !is.na(w$post), ]
  # a stretch is clean if no change falls on games n - k + 1 .. n + k of the same club-season
  near <- unlist(lapply(seq_len(nrow(changes)), function(r) {
    paste(changes$season[r], changes$club[r], (changes$game[r] - k):(changes$game[r] + k - 1))
  }))
  control <- pool[!(paste(pool$season, pool$club, pool$n) %in% near), ]
  rownames(control) <- NULL
  list(treated = treated, control = control)
}

sp <- split_windows(tg, changes, K)
treated <- sp$treated
control <- sp$control
E <- pub$effect
cat(sprintf("%d changes with full windows; %d control stretches\n", nrow(treated), nrow(control)))
cat(sprintf("before %.3f, after %.3f points per game\n", mean(treated$pre) / K, mean(treated$post) / K))
cat(sprintf("improved: %.1f%%\n", 100 * mean(treated$post > treated$pre)))
stopifnot(nrow(treated) == pub$summary$analysed, nrow(control) == pub$summary$controlPositions)
near(mean(treated$pre) / K, E$prePpg); near(mean(treated$post) / K, E$postPpg)
near(mean(treated$post > treated$pre), E$improvedShare)

## 4. Regression to the mean, without any coach change

Take every six-game stretch where the coach stayed. How many points per game followed, by
how many points the team had just taken? If form simply continued, the circles would sit on the
diagonal.

In [ ]:
depth_mean <- tapply(control$post, control$pre, mean)
depth_n <- tapply(control$post, control$pre, length)
depth_pre <- as.integer(names(depth_mean))
plot(depth_pre, depth_mean / K, cex = sqrt(depth_n / max(depth_n)) * 3, xlim = c(0, 18),
     ylim = c(0, 3), bty = "l", las = 1, xlab = "Points in the previous 6 games",
     ylab = "Points per game, next 6 games",
     main = "Figure 1: extreme runs are followed by less extreme ones", adj = 0, cex.main = 1)
abline(0, 1 / 6, lty = 2)
abline(h = mean(tg$pts), lty = 3)
points(treated$pre, treated$post / K, col = "#c0392b", cex = 0.6)
legend("topleft", c("coach stayed", "coach changed"), pch = 1, col = c("black", "#c0392b"),
       bty = "n")

slump <- control[control$pre <= 4, ]
cat(sprintf("stretches with <= 4 points: %.3f points per game next (n = %d)\n",
            mean(slump$post) / K, nrow(slump)))
cat(sprintf("0 points from 6: %.3f; 18 points from 6: %.3f\n",
            depth_mean[["0"]] / K, depth_mean[["18"]] / K))
near(mean(slump$post) / K, pub$slump$postPpg)
for (row in pub$depth) near(depth_mean[[as.character(row$pre)]] / K, row$postPpg)
cat(sprintf("%.0f%% of changes came after 5 points or fewer\n", 100 * mean(treated$pre <= 5)))

## 5. The fair comparison

For each change, take all control stretches with **exactly the same points in the previous six
games** and average their next six. The effect of a change is its own next-six points minus
that average, divided by six (points per game).

In [ ]:
# Mean of post minus the matched controls' mean post, per game.
effect <- function(treated, control, k) {
  cm <- tapply(control$post, control$pre, mean)
  cp <- cm[as.character(treated$pre)]
  ok <- !is.na(cp)
  mean(treated$post[ok] - cp[ok]) / k
}

cm <- tapply(control$post, control$pre, mean)
ctrl_post <- cm[as.character(treated$pre)]
eff <- effect(treated, control, K)
cat(sprintf("matched controls: %.3f points per game after\n", mean(ctrl_post) / K))
cat(sprintf("difference of the means: %+.3f points per game\n", eff))
ctrl_improved <- mean(tapply(control$post > control$pre, control$pre, mean)[as.character(treated$pre)])
cat(sprintf("matched controls that improved: %.1f%%\n", 100 * ctrl_improved))
near(mean(ctrl_post) / K, E$controlPostPpg)
near(ctrl_improved, E$controlImprovedShare)
cat(sprintf("opponents after: %.3f (changes) vs %.3f (controls)\n",
            mean(treated$opp_post), mean(control$opp_post)))
near(mean(treated$opp_post), E$oppPostPpg); near(mean(control$opp_post), E$controlOppPostPpg)

### How uncertain is that?

The same comparison as a regression: points per game in the six games after, on a dummy for the
coach change and one dummy for each points total in the six games before. The coefficient of the
change dummy is the effect; its t-test gives the p-value and the 95% confidence interval. The
t-test treats all stretches as independent; overlapping stretches of the same club are not quite.
The next cell is `notebooks/coach_firing_stats.R`.

In [ ]:
# coach_firing_stats.R
# ──────────────────────────────────────────────────────────────────────
# The test of story #001 for one calculation path: a linear model with a t-test. The pipeline and
# the notebook use the same function, so they produce identical numbers.
#
#   matched:    post points per game ~ coach change (dummy) + one dummy per points before.
#               Changes and control stretches with the same points before are compared, the
#               coefficient of the change dummy is the effect.
#   before:     (post - pre) points per game of the changes alone, t-test against 0
#               (an intercept-only model).
#   all_slumps: (post - pre) points per game ~ coach change (dummy), changes in a slump and all
#               control stretches in a slump (at most 2k/3 points in k games).
#
# With opponents = "yes" the outcome is points above an expectation from the opponent's strength
# (pre_adj, post_adj); the matching dummies stay on the raw points before.
#
# Requires: base R only.
# ──────────────────────────────────────────────────────────────────────

ALPHA <- 0.05

# Effect (points per game), standard error, residual degrees of freedom, 95% interval and
# two-sided p-value of the t-test, the number of coach changes and the fitted model.
test_path <- function(treated, control, comparison, opponents, k) {
  pre <- if (opponents == "yes") "pre_adj" else "pre"
  post <- if (opponents == "yes") "post_adj" else "post"
  if (comparison == "before") {
    d <- data.frame(y = (treated[[post]] - treated[[pre]]) / k)
    fit <- lm(y ~ 1, data = d)
    term <- "(Intercept)"
  } else if (comparison == "matched") {
    d <- rbind(
      data.frame(y = treated[[post]] / k, change = 1, bin = treated$pre),
      data.frame(y = control[[post]] / k, change = 0, bin = control$pre)
    )
    d <- d[d$bin %in% control$pre, ]
    fit <- lm(y ~ change + factor(bin), data = d)
    term <- "change"
  } else {
    slump <- function(df) df[df$pre <= (2 * k) %/% 3, ]
    t <- slump(treated)
    c <- slump(control)
    d <- rbind(
      data.frame(y = (t[[post]] - t[[pre]]) / k, change = 1),
      data.frame(y = (c[[post]] - c[[pre]]) / k, change = 0)
    )
    fit <- lm(y ~ change, data = d)
    term <- "change"
  }
  co <- summary(fit)$coefficients
  est <- co[term, "Estimate"]
  se <- co[term, "Std. Error"]
  df <- fit$df.residual
  q <- qt(1 - ALPHA / 2, df)
  n <- if (comparison == "before") nrow(d) else sum(d$change == 1)
  list(effect = est, se = se, df = df, ci_low = est - q * se, ci_high = est + q * se,
       p = co[term, "Pr(>|t|)"], n = n, fit = fit)
}

sp6 <- list(treated = treated, control = control)
m <- test_path(treated, control, "matched", "no", K)
print(summary(m$fit)$coefficients["change", ])
near(m$effect, E$perGame)
# the same coefficient from lm() written out by hand
d6 <- rbind(data.frame(y = treated$post / K, change = 1, bin = treated$pre),
            data.frame(y = control$post / K, change = 0, bin = control$pre))
stopifnot(abs(coef(lm(y ~ change + factor(bin), data = d6))[["change"]] - m$effect) < 1e-12)

In [ ]:
# Points per game at relative games -k..-1 and +1..+k, controls matched on pre.
trajectories <- function(tg, treated, control, k) {
  pkey <- paste(tg$season, tg$club, tg$n)
  vec <- function(df) {
    sapply(-k:(k - 1), function(j) tg$pts[match(paste(df$season, df$club, df$n + j), pkey)])
  }
  t <- vec(treated)
  cv <- vec(control)
  by_pre <- rowsum(cv, control$pre) / as.vector(table(control$pre))
  c <- by_pre[as.character(treated$pre), , drop = FALSE]
  list(t = t, c = c)
}

traj <- trajectories(tg, treated, control, K)
t_traj <- colMeans(traj$t, na.rm = TRUE)
c_traj <- colMeans(traj$c, na.rm = TRUE)
for (i in seq_along(t_traj)) near(t_traj[i], pub$trajectory$changes[[i]])
for (i in seq_along(c_traj)) near(c_traj[i], pub$trajectory$controls[[i]])
cat(sprintf("last game before the change: %.3f points on average\n", t_traj[K]))

rel <- c(-K:-1, 1:K)
x <- c(1:K, (K + 2):(2 * K + 1))
plot(x, t_traj, type = "n", ylim = c(0, 3), bty = "l", las = 1, xaxt = "n",
     xlab = "Games before / after the change", ylab = "Points per game",
     main = "Figure 2: the bounce, with and without a coach change", adj = 0, cex.main = 1)
axis(1, at = x, labels = sprintf("%+d", rel))
abline(v = K + 1, lty = 3)
for (h in list(1:K, (K + 1):(2 * K))) {
  lines(x[h], c_traj[h], type = "o", lty = 2)
  lines(x[h], t_traj[h], type = "o", col = "#c0392b")
}
legend("topleft", c("coach changed", "same slump, coach stayed"), lty = c(1, 2),
       col = c("#c0392b", "black"), bty = "n")

## 6. Every calculation path

Three choices decide the answer: what to compare with, how many games to look at, and whether
the opponents' strength counts. 3 × 3 × 2 = 18 paths. Each is recomputed here and must equal
the published table.

In [ ]:
FORK_OPTIONS <- list(
  comparison = c("before", "matched", "all_slumps"),
  window = c("5", "6", "10"),
  opponents = c("no", "yes")
)

specs <- list()
for (comparison in FORK_OPTIONS$comparison) for (window in FORK_OPTIONS$window)
  for (opponents in FORK_OPTIONS$opponents) {
    k <- as.integer(window)
    sp <- split_windows(tg, find_changes(tg, k)$changes, k)
    s <- test_path(sp$treated, sp$control, comparison, opponents, k)
    specs[[length(specs) + 1]] <- c(list(choices = list(comparison = comparison, window = window,
                                                        opponents = opponents)),
                                    s[c("effect", "ci_low", "ci_high", "p", "n")])
  }
for (i in seq_along(specs)) {
  r <- specs[[i]]
  p <- pub$specs[[i]]
  stopifnot(identical(r$choices, p$choices), r$n == p$n)
  for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(r[[k]] - p[[k]]) < 1e-9)
}
print(do.call(rbind, lapply(specs, function(s) data.frame(s$choices, effect = s$effect,
  ci_low = s$ci_low, ci_high = s$ci_high, p = s$p, n = s$n))), digits = 3)
yes <- sum(vapply(specs, function(s) s$p < ALPHA && s$effect > 0, TRUE))
cat(sprintf("the answer is yes on %d of %d paths\n", yes, length(specs)))

crisis <- treated[treated$pre <= 4, ]
cat(sprintf("crisis changes: %.3f → %.3f points per game\n",
            mean(crisis$pre) / K, mean(crisis$post) / K))
near(mean(crisis$pre) / K, pub$slump$crisisPrePpg); near(mean(crisis$post) / K, pub$slump$crisisPostPpg)

## 7. The result table

The table under "The hypothesis test" on the website comes from the pipeline: the coefficient of
the coach-change dummy in the main path (matched, six games, no opponent adjustment) with its
standard error, t-interval and p-value.

In [ ]:
co <- summary(m$fit)$coefficients["change", ]
q <- qt(0.975, m$df)
mine <- c(co[["Estimate"]], co[["Std. Error"]], co[["Estimate"]] - q * co[["Std. Error"]],
          co[["Estimate"]] + q * co[["Std. Error"]], co[["Pr(>|t|)"]])
stopifnot(max(abs(mine - unlist(pub$estimation$rows[[1]]$values))) < 1e-9)
names(mine) <- c("Coefficient", "Std. Error", "Lower 95%", "Upper 95%", "Prob.")
print(round(mine, 6))

## Data and methods

* **Sources.** transfermarkt-datasets, `games` table (CC0 1.0; release of 5 September 2026; data
  from Transfermarkt.de): <https://github.com/dcaribou/transfermarkt-datasets>.
  openfootball/deutschland (CC0 1.0): <https://github.com/openfootball/deutschland>.
* **Coverage.** Bundesliga 2012/13–2025/26, 4,284 matches. Results agree between both sources
  on every match date.
* **Definitions.** Coach change = first league game under a different manager within a season;
  stand-ins excluded; caretaker + permanent coach = one change. The data do not say whether a
  coach was dismissed or left, so all changes are included.
* **Comparison.** Main calculation: six-game stretches with no coach change in the twelve-game
  window, matched exactly on points in the previous six games. Alternatives: the team before the
  change; all no-change stretches with at most ⌊2K/3⌋ points (difference in differences);
  windows of 5 and 10 games; points above an expectation from the opponent's season points per
  game and home advantage. The opponent's season figure includes games after the change.
* **Uncertainty.** Least-squares regression with a coach-change dummy and dummies for the points
  before; t-test of the change dummy, 95% t-interval.
* **Fixed before the coach data:** hypothesis, 6-game window, matched controls. Fixed after:
  the other windows, the opponent fork, the all-slumps comparison, the smallest relevant effect
  (0.15 points per game) and the test.
* **Prior work.** Heuer, A. et al. (2011), *PLoS ONE* 6(3), e17664; ter Weel, B. (2011),
  *De Economist* 159, 279–303.
* **Code.** `notebooks/bundesliga_matches_pipeline.R`, `notebooks/coach_firing_pipeline.R`,
  `notebooks/coach_firing_stats.R` (base R and jsonlite).

In [ ]:
cat("All statistics reproduced ✓\n")